In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import scipy.stats as stats

tickers = ["NVDA", "GS", "BTC-USD"]
portfolio_value = 1_000_000 
confidence_level = 0.95
alpha = 1 - confidence_level

print("Fetching historical data from Yahoo Finance.")
raw_data = yf.download(
    tickers, start="2023-10-01", end="2026-10-01", auto_adjust=True
)
data = raw_data["Close"].dropna()

In [ ]:
# Calculate daily log returns
returns = np.log(data / data.shift(1)).dropna()

# Define portfolio weights (Must sum to 1.0)
weights = np.array([0.40, 0.40, 0.20]) 

# Calculate the single daily return sequence for the entire portfolio
portfolio_returns = returns.dot(weights) 

print("\n--- Asset Correlation Matrix ---")
print(returns.corr())

print("\n--- Worst 5 Daily Returns of Your Portfolio (%) ---")
print(portfolio_returns.sort_values().head() * 100)

portfolio_value = 1_000_000  # Assume a hypothetical $1,000,000 portfolio
confidence_level = 0.95      # 95% risk horizon (the worst 5% of days)

# Sort your returns from worst to best
sorted_returns = np.sort(portfolio_returns)

# Find the threshold boundary for the worst 5% of days
alpha = 1 - confidence_level
historical_var_pct = -np.percentile(sorted_returns, alpha * 100)
historical_var_usd = historical_var_pct * portfolio_value

In [ ]:
# Expected Shortfall (ES): Average loss on the days that broke past the VaR barrier
historical_es_pct = -sorted_returns[portfolio_returns <= -historical_var_pct].mean()
historical_es_usd = historical_es_pct * portfolio_value

print("--- Finalized Historical Risk Metrics ---")
print(f"95% Daily VaR (Percent): {historical_var_pct * 100:.2f}%")
print(f"95% Daily VaR (Dollar):  ${historical_var_usd:,.2f}")
print(f"95% Daily ES (Percent):  {historical_es_pct * 100:.2f}%")
print(f"95% Daily ES (Dollar):   ${historical_es_usd:,.2f}")

# Return to a Student's t-distribution
df, loc, scale = stats.t.fit(portfolio_returns)

# Risk parameter
confidence_level = 0.95
alpha = 1 - confidence_level

# t-distribution
t_var_pct = -stats.t.ppf(alpha, df, loc, scale)

# Print outputs
print("--- Custom Student's t-Distribution Results ---")
print(f"True Degrees of Freedom (Fat-Tail Metric): {df:.2f}")
print(f"Custom Student's t-VaR: {t_var_pct * 100:.2f}%")


In [ ]:
# Fix historical ES
true_tail_losses = sorted_returns[sorted_returns <= -historical_var_pct]
corrected_historical_es_pct = -true_tail_losses.mean()
corrected_historical_es_usd = corrected_historical_es_pct * portfolio_value

# Decision
parametric_t_shocks = stats.t.rvs(df, loc=loc, scale=scale, size=100_000)
parametric_t_es_pct = -parametric_t_shocks[parametric_t_shocks <= stats.t.ppf(alpha, df, loc, scale)].mean()
parametric_t_es_usd = parametric_t_es_pct * portfolio_value

print("--- Corrected & Advanced Risk Metrics ---")
print(f"Corrected Historical ES: {corrected_historical_es_pct * 100:.2f}% (${corrected_historical_es_usd:,.2f})")
print(f"Parametric Student's t-ES: {parametric_t_es_pct * 100:.2f}% (${parametric_t_es_usd:,.2f})")

In [ ]:
# Count times the actual historical losses exceeded Student's t-VaR
exceptions = portfolio_returns < -t_var_pct
n_exceptions = np.sum(exceptions)
total_days = len(portfolio_returns)

# The expected exception rate
p_expected = 1 - confidence_level
observed_rate = n_exceptions / total_days

# Kupiec Likelihood Ratio 
lr_numerator = (p_expected ** n_exceptions) * ((1 - p_expected) ** (total_days - n_exceptions))
lr_denominator = (observed_rate ** n_exceptions) * ((1 - observed_rate) ** (total_days - n_exceptions))
lr_statistic = -2 * np.log(lr_numerator / lr_denominator)

#Chi-Square critical value threshold
critical_value = stats.chi2.ppf(0.95, df=1)

print("--- Model Validation: Kupiec POF Backtest ---")
print(f"Total Portfolio Trading Days: {total_days}")
print(f"Expected Number of Exceptions: {int(total_days * p_expected)}")
print(f"Observed Number of Exceptions: {n_exceptions}")
print(f"Kupiec Likelihood Ratio: {lr_statistic:.4f} (Critical Threshold: {critical_value:.4f})")

if lr_statistic > critical_value:
    print("Verdict: REJECT the model. The model is statistically inaccurate.")
else:
    print("Verdict: FAIL TO REJECT the model. The model is structurally sound.")